# 金融问答 RAG 实验笔记本

基于 FinanceBench 数据集的 RAG 系统实验，支持：
- 多检索器（BM25 稀疏检索、Contriever 稠密检索、Hybrid RRF 混合检索）
- 多生成器（阿里百炼 API、本地 Llama 模型）
- 幻觉评估（FActScore、Self-RAG）
- 检索-生成对比实验


## 1. 环境设置与依赖安装

In [ ]:
# 安装所有依赖
!pip install datasets numpy torch rank-bm25 transformers openai httpx python-dotenv tqdm bitsandbytes>=0.46.1 --quiet

In [ ]:
## 4. 初始化实验组件

# 包括：数据加载、检索器（BM25/Contriever/HybridRRF）、生成器、评估器
from methods import (
    load_financebench, BM25Retriever, ContrieverRetriever, HybridRRFRetriever,
    RAGSystem, QwenGenerator, collect_retrieval_results, evaluate, evaluate_with_retrieval_cache
)
from eval import FActScoreEvaluator
import torch
import gc

# 加载数据
print("加载数据...")
examples, corpus = load_financebench(subset_size=SUBSET)

# 初始化 BM25 检索器（CPU）
print("初始化 BM25 检索器...")
bm25_retriever = BM25Retriever(corpus, cache_path="./cache/bm25_index_finance.pkl")

# 初始化 Contriever 检索器（GPU/CPU）
print("初始化 Contriever 检索器...")
try:
    contriever_retriever = ContrieverRetriever(
        corpus,
        cache_path="./cache/contriever_embeddings_finance.pkl",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print("✅ Contriever 检索器初始化完成")
except Exception as e:
    print(f"⚠️ Contriever 初始化失败: {type(e).__name__}: {e}")
    contriever_retriever = None

# 初始化 Hybrid RRF 混合检索器
print("初始化 Hybrid RRF 混合检索器...")
if contriever_retriever is not None:
    hybrid_retriever = HybridRRFRetriever(
        documents=corpus,
        bm25_retriever=bm25_retriever,
        contriever_retriever=contriever_retriever
    )
    print(f"✅ Hybrid RRF 混合检索器初始化完成")
else:
    hybrid_retriever = None
    print("⚠️ Hybrid RRF 初始化跳过（Contriever 不可用）")

# 清理显存（为后续模型预留空间）
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        torch.cuda.ipc_collect()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"✅ 显存清理完成，可用显存: {free_gb:.2f} GB")

# 初始化 API 生成器
print("初始化 API 生成器...")
generator = QwenGenerator(model=MODEL_NAME)

# 初始化 FActScore 评估器（API）
print("初始化 FActScore 评估器...")
factscore_evaluator = FActScoreEvaluator(model=MODEL_NAME)

# 创建 RAG 系统（使用 BM25 作为默认检索器）
print("创建 RAG 系统...")
rag = RAGSystem(bm25_retriever, generator, top_k=TOP_K)

print("✅ 初始化完成！")


In [ ]:
# 检查 GPU 是否可用
!python -c "import torch; print(f'✅ CUDA 可用: {torch.cuda.is_available()}')"

## 2. 从 GitHub 同步代码

In [ ]:
import time
import os
import sys
import subprocess

timestamp = int(time.time())
BRANCH = "2_domain_retrival"
BASE_RAW = f"https://raw.githubusercontent.com/kitee0918/-nlp-rag-project/{BRANCH}"
TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()  # 私有仓库时可在环境变量里设置

# 清除旧文件和缓存
!rm -f methods.py main.py eval.py llm.py retriever.py prompt.py
!rm -rf __pycache__
!rm -rf cache/

# 清除已导入模块
for module_name in ['methods', 'main', 'eval', 'llm', 'retriever', 'prompt']:
    if module_name in sys.modules:
        del sys.modules[module_name]

def download_raw(filename):
    url = f"{BASE_RAW}/{filename}?t={timestamp}"
    cmd = ["wget"]
    if TOKEN:
        cmd += ["--header", f"Authorization: token {TOKEN}"]
    cmd += [url, "-O", filename]
    subprocess.run(cmd, check=True)

for f in ['methods.py', 'main.py', 'eval.py', 'llm.py', 'retriever.py', 'prompt.py']:
    download_raw(f)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

print(f"✅ {timestamp}: 代码已更新（methods/main/eval/llm/retriever/prompt），可直接重新运行实验")

## 3. 配置信息

In [ ]:
import os

# API 配置常量
API_KEY = "sk-d1a2d27081934dd9b556b327869bb7cc"
MODEL_NAME = "qwen2.5-14b-instruct-1m"  # 使用14B版本模型

# 设置 API Key
os.environ['DASHSCOPE_API_KEY'] = API_KEY

# 实验参数
SUBSET = 100
TOP_K = 3

## 4. 初始化实验组件

包括：数据加载、检索器（BM25/Contriever）、生成器、评估器

In [ ]:
from methods import load_financebench, BM25Retriever, ContrieverRetriever, RAGSystem, QwenGenerator
from eval import FActScoreEvaluator
import torch
import gc

# 加载数据
print("加载数据...")
examples, corpus = load_financebench(subset_size=SUBSET)

# 初始化 BM25 检索器（CPU）
print("初始化 BM25 检索器...")
bm25_retriever = BM25Retriever(corpus, cache_path="./cache/bm25_index_finance.pkl")

# 初始化 Contriever 检索器（GPU/CPU）
print("初始化 Contriever 检索器...")
try:
    contriever_retriever = ContrieverRetriever(
        corpus,
        cache_path="./cache/contriever_embeddings_finance.pkl",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print("✅ Contriever 检索器初始化完成")
except Exception as e:
    print(f"⚠️ Contriever 初始化失败: {type(e).__name__}: {e}")
    contriever_retriever = None

# 清理显存（为后续模型预留空间）
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    if hasattr(torch.cuda, 'ipc_collect'):
        torch.cuda.ipc_collect()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"✅ 显存清理完成，可用显存: {free_gb:.2f} GB")

# 初始化 API 生成器
print("初始化 API 生成器...")
generator = QwenGenerator(model=MODEL_NAME)

# 初始化 FActScore 评估器（API）
print("初始化 FActScore 评估器...")
factscore_evaluator = FActScoreEvaluator(model=MODEL_NAME)

# 创建 RAG 系统（使用 BM25 作为默认检索器）
print("创建 RAG 系统...")
rag = RAGSystem(bm25_retriever, generator, top_k=TOP_K)

print("✅ 初始化完成！")

In [ ]:
## 5. 单样本测试 Hybrid RRF

# 演示 Hybrid RRF 混合检索器如何融合 BM25 和 Contriever 的检索结果
# 测试单次检索对比
if hybrid_retriever is not None:
    test_query = examples[0]['question']
    print(f"测试查询: {test_query[:80]}...")
    print("="*60)
    
    # 获取各检索器结果
    results = hybrid_retriever.get_individual_results(test_query, top_k=TOP_K)
    
    print(f"\n📊 BM25 Top-{TOP_K} 结果:")
    for i, (doc, score) in enumerate(results['bm25'], 1):
        print(f"  {i}. [{doc['id']}] 分数: {score:.4f}")
    
    print(f"\n📊 Contriever Top-{TOP_K} 结果:")
    for i, (doc, score) in enumerate(results['contriever'], 1):
        print(f"  {i}. [{doc['id']}] 分数: {score:.4f}")
    
    print(f"\n📊 Hybrid RRF Top-{TOP_K} 结果 (k=60):")
    for i, (doc, score) in enumerate(results['hybrid'], 1):
        print(f"  {i}. [{doc['id']}] RRF分数: {score:.4f}")
    
    # 使用静态方法直接融合
    print(f"\n🔧 使用静态方法直接融合:")
    fused = HybridRRFRetriever.fuse_results(
        results['bm25'], results['contriever'], k=60, top_k=TOP_K
    )
    for i, (doc, score) in enumerate(fused, 1):
        print(f"  {i}. [{doc['id']}] RRF分数: {score:.4f}")
else:
    print("⚠️ Hybrid RRF 检索器不可用，跳过测试")


In [ ]:
## 6. 三种检索器对比实验

# 使用缓存优化机制，避免 Hybrid RRF 重复检索
from tqdm import tqdm

# 定义评估函数
def run_retriever_comparison(examples, retrievers_dict, generator, top_k=3):
    """
    对比多种检索器的性能
    
    使用缓存优化：先收集 BM25 和 Contriever 结果，Hybrid 直接复用
    """
    results = {}
    
    # 检查是否需要预收集（当三种检索器都存在时）
    retriever_names = list(retrievers_dict.keys())
    if 'BM25' in retriever_names and 'Contriever' in retriever_names and 'HybridRRF' in retriever_names:
        print("📦 预收集 BM25 和 Contriever 检索结果（避免 Hybrid 重复检索）")
        
        # 收集检索结果
        fetch_k = top_k * 2
        bm25_cache = collect_retrieval_results(retrievers_dict['BM25'], examples, top_k=fetch_k)
        contriever_cache = collect_retrieval_results(retrievers_dict['Contriever'], examples, top_k=fetch_k)
        
        retrieval_cache = {'bm25': bm25_cache, 'contriever': contriever_cache}
        print(f"✅ 已收集 {len(bm25_cache)} 个查询的检索结果缓存")
    else:
        retrieval_cache = None
    
    # 评估每种检索器
    for name, retriever in retrievers_dict.items():
        print(f"\n{'='*60}")
        print(f"🔬 评估: {name}")
        print(f"{'='*60}")
        
        rag = RAGSystem(retriever, generator, top_k=top_k)
        
        # 对于 Hybrid 使用缓存
        if name == 'HybridRRF' and retrieval_cache is not None:
            eval_result = evaluate_with_retrieval_cache(rag, examples, retrieval_cache)
        else:
            eval_result = evaluate(rag, examples)
        
        results[name] = eval_result
        
        print(f"\n📊 {name} 评估结果:")
        print(f"   F1 Score:    {eval_result['f1_score']:.4f}")
        print(f"   Recall@K:   {eval_result['recall@k']:.4f}")
        print(f"   Precision@K: {eval_result['precision@k']:.4f}")
        print(f"   Hit@K:      {eval_result['hit@k']:.4f}")
    
    return results

# 构建检索器字典
retrievers_dict = {'BM25': bm25_retriever}
if contriever_retriever is not None:
    retrievers_dict['Contriever'] = contriever_retriever
if hybrid_retriever is not None:
    retrievers_dict['HybridRRF'] = hybrid_retriever

# 运行对比实验
print("\n" + "="*70)
print("🚀 开始三种检索器对比实验")
print("="*70)

comparison_results = run_retriever_comparison(
    examples, retrievers_dict, generator, top_k=TOP_K
)


In [ ]:
## 7. 对比结果展示
import pandas as pd

# 创建结果表格
results_data = []
for name, result in comparison_results.items():
    results_data.append({
        '检索器': name,
        'F1 Score': result['f1_score'],
        'Recall@K': result['recall@k'],
        'Precision@K': result['precision@k'],
        'Hit@K': result['hit@k']
    })

df_results = pd.DataFrame(results_data)
print("\n📊 对比结果表格:")
print(df_results.to_string(index=False))

# 找出最佳检索器
best_hit = max(comparison_results.keys(), key=lambda k: comparison_results[k]['hit@k'])
best_f1 = max(comparison_results.keys(), key=lambda k: comparison_results[k]['f1_score'])

print(f"\n🏆 最佳 Hit@K:    {best_hit} ({comparison_results[best_hit]['hit@k']:.4f})")
print(f"🏆 最佳 F1 Score: {best_f1} ({comparison_results[best_f1]['f1_score']:.4f})")

# 计算 Hybrid 相对 BM25 的提升
if 'HybridRRF' in comparison_results and 'BM25' in comparison_results:
    hit_improve = (comparison_results['HybridRRF']['hit@k'] - comparison_results['BM25']['hit@k']) / comparison_results['BM25']['hit@k'] * 100 if comparison_results['BM25']['hit@k'] > 0 else 0
    f1_improve = (comparison_results['HybridRRF']['f1_score'] - comparison_results['BM25']['f1_score']) / comparison_results['BM25']['f1_score'] * 100 if comparison_results['BM25']['f1_score'] > 0 else 0
    
    print(f"\n📈 Hybrid RRF vs BM25 提升:")
    print(f"   Hit@K:    {'+' if hit_improve >= 0 else ''}{hit_improve:.2f}%")
    print(f"   F1 Score: {'+' if f1_improve >= 0 else ''}{f1_improve:.2f}%")


In [ ]:
## 8. 使用 run_comparison_experiment 一键运行

也可以使用封装好的函数一键运行完整实验
from methods import run_comparison_experiment

# 方式1: 对比所有检索器
print("\n🚀 使用 run_comparison_experiment 一键运行...")
results = run_comparison_experiment(
    subset_size=SUBSET,
    top_k=TOP_K,
    model=MODEL_NAME,
    use_retrievers=['bm25', 'contriever', 'hybrid']
)
